# Eco-City PPO — Google Colab (GPU)

1. **Runtime → Change runtime type → GPU** (e.g. A100 if available).
2. Run the **clone** cell below once per session. It pulls the repo from GitHub and sets `PROJECT_ROOT`.

Repository: [https://github.com/Shreya-Mendi/Eco-city](https://github.com/Shreya-Mendi/Eco-city)

In [ ]:
# Clone the repo from GitHub (Colab) and checkout the right branch
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Shreya-Mendi/Eco-city.git"
BRANCH = "colab-local-handoff"  # branch with terrain + eval + tuning
CLONE_PARENT = Path("/content")
PROJECT_ROOT = CLONE_PARENT / "Eco-city"

if not (PROJECT_ROOT / "requirements.txt").is_file():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Repo already exists; fetching + checking out", BRANCH)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("CWD:", os.getcwd())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"]).decode().strip())

## Hyperparameters (edit these)

| Setting | Default | Notes |
|---------|---------|--------|
| `TOTAL_TIMESTEPS` | 500_000 | Smoke test: 50_000–100_000 |
| `SAVE_PATH` | `results/ppo_eco_city` | SB3 writes `SAVE_PATH.zip` |
| `DEVICE` | `"auto"` | Colab GPU: usually `"cuda"`; `"cpu"` to force CPU |
| `EXPORT_JSON` | `True` | Writes `city_data.json` for the Three.js viewer |

In [ ]:
# Optional: persist to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/Eco-city")
# os.chdir(PROJECT_ROOT)
# sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from env.city_env import CityEnv
from stable_baselines3.common.env_checker import check_env

_env = CityEnv()
check_env(_env, warn=True)
print("ENV OK")

In [ ]:
# Run unit tests (terrain, buildable mask, obs shape)
!pip install -q pytest
!pytest tests/ -q

In [ ]:
from pathlib import Path

from agents.ppo_agent import make_ppo_agent, save, train
from training.callbacks import MetricsCallback
from training.vec_env import make_vec_train_env, save_vecnormalize, unwrap_city_env, vecnorm_save_path
from visualization.exporter import export

# --- knobs ---
TOTAL_TIMESTEPS = 500_000
LOGDIR = Path("logs/ppo")
SAVE_PATH = "results/ppo_eco_city"
DEVICE = "auto"  # "cuda" | "cpu" | "auto"
EXPORT_JSON = True

LOGDIR.mkdir(parents=True, exist_ok=True)
Path("results").mkdir(parents=True, exist_ok=True)
Path("logs").mkdir(parents=True, exist_ok=True)

vec_env = make_vec_train_env(log_csv=LOGDIR / "monitor.csv")
base = unwrap_city_env(vec_env)

device_kw = {}
if DEVICE != "auto":
    device_kw["device"] = DEVICE

model = make_ppo_agent(vec_env, tensorboard_log="logs", **device_kw)
callback = MetricsCallback()

train(model, total_timesteps=TOTAL_TIMESTEPS, callback=callback)
save(model, SAVE_PATH)
save_vecnormalize(vec_env, vecnorm_save_path(SAVE_PATH))

if EXPORT_JSON:
    # Deterministic rollout so exporter writes a clean trajectory (see train.py)
    from training.train import _rollout_one_episode_for_export
    base._export_traj = []
    _rollout_one_episode_for_export(model, vec_env, seed=42)
    export(base, "visualization/threejs/city_data.json")
    base._export_traj = None

print("Done. Model:", SAVE_PATH + ".zip")
print("VecNormalize:", vecnorm_save_path(SAVE_PATH))

## Evaluate: PPO vs baselines
Quick summary first (`eval_summary.json`), then the full 4-experiment suite (`experiment_results.json`).

In [ ]:
# Quick eval: PPO vs random / greedy / heuristic
import json, subprocess
subprocess.run(["python", "evaluation/eval_suite.py", "--quick", "--model", SAVE_PATH], check=True)
with open("results/eval_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Full experiment suite (4 experiments). Writes results/experiment_results.json.
import subprocess
subprocess.run(["python", "evaluation/experiments.py"], check=True)

## Hyperparameter tuning (optional)
Runs a small grid (lr / ent_coef / batch_size). Takes ~4× `TUNE_TIMESTEPS` training time.
Best run is copied to `results/ppo_eco_city_tuned.zip`.

In [ ]:
# Hyperparameter sweep (4 configs). Writes results/tuning_results.json + ppo_eco_city_tuned.zip
TUNE_TIMESTEPS = 80_000
import json, subprocess
subprocess.run(["python", "training/tune_ppo.py", "--timesteps", str(TUNE_TIMESTEPS)], check=True)
with open("results/tuning_results.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Save top-K rollouts for the 3D viewer
Runs `NUM_ROLLOUTS` deterministic rollouts with different seeds, keeps the top `TOP_K` by return, writes them to `visualization/threejs/rollouts/` with an `index.json` so the viewer can switch between them.

In [ ]:
NUM_ROLLOUTS = 20
TOP_K = 3
import json, subprocess
subprocess.run([
    "python", "evaluation/save_top_rollouts.py",
    "--model", SAVE_PATH,
    "--episodes", str(NUM_ROLLOUTS),
    "--top-k", str(TOP_K),
], check=True)
with open("visualization/threejs/rollouts/index.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Download artifacts (Colab)
Run after training. Download **both** the policy zip and the VecNormalize stats (evaluation needs both).

In [ ]:
try:
    from google.colab import files
    from pathlib import Path
    import shutil
    from training.vec_env import vecnorm_save_path
    save = "results/ppo_eco_city"

    rollouts_dir = Path("visualization/threejs/rollouts")
    rollouts_zip = None
    if rollouts_dir.is_dir() and any(rollouts_dir.glob("run_*.json")):
        rollouts_zip = shutil.make_archive("results/top_rollouts", "zip", root_dir=str(rollouts_dir))

    wanted = [
        save + ".zip",
        str(vecnorm_save_path(save)),
        "visualization/threejs/city_data.json",
        "results/eval_summary.json",
        "results/experiment_results.json",
        "results/tuning_results.json",
        "results/ppo_eco_city_tuned.zip",
        str(vecnorm_save_path("results/ppo_eco_city_tuned")),
        rollouts_zip,
    ]
    for path in wanted:
        if path and Path(path).is_file():
            files.download(path)
        else:
            print("skip (not found):", path)
except ImportError:
    print("Not on Colab — artifacts are under results/ and visualization/threejs/")

## TensorBoard (optional)
In Colab: `%load_ext tensorboard` then `%tensorboard --logdir logs`